In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"


In [0]:
tables = spark.sql(f"SHOW TABLES IN {silver_schema}")
display(tables)

##gold.dim_motivo

In [0]:
#gold.dim_motivo (origem silver.tb_motivo)

# partição única explícita: a dimensão é pequena, então juntar tudo numa partição não pesa
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_motivo").cast("int"))

df_silver_motivo = spark.read.table(f"{silver_schema}.tb_base_motivos")

df_dim_motivo = (
    df_silver_motivo
    .withColumn("SK_motivo", F.row_number().over(w_sk).cast("bigint"))
    .select(
        F.col("SK_motivo"),
        F.col("id_motivo").cast("int").alias("id_motivo"),
        F.col("nome_motivo").cast("string").alias("nome"),
        F.col("categoria").cast("string").alias("categoria"),
        F.col("criticidade").cast("string").alias("criticidade"),
    )
    .orderBy("SK_motivo")
)

display(df_dim_motivo)

df_dim_motivo.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_motivo")

## gold.dim_canal

In [0]:
#gold.dim_canal (origem silver.tb_canais)

w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_canal").cast("int"))

df_silver_canais = spark.read.table(f"{silver_schema}.tb_canais")

# a silver não traz etapa_funil nem nivel_etapa: derivamos dos valores padronizados na Silver
k_nome_canal = F.lower(F.trim(F.col("nome_canal")))

etapa_funil = (
    F.when(k_nome_canal == "ura", F.lit("Autosserviço"))
    .when(k_nome_canal == "chatbot", F.lit("Autosserviço"))
    .when(k_nome_canal == "atendimento inicial", F.lit("Atendimento Inicial"))
    .when(k_nome_canal == "atendimento especializado", F.lit("Atendimento Especializado"))
    .otherwise(F.lit("Desconhecido"))
)

# ordem manual, para refletir a posição da etapa no funil de atendimento
nivel_etapa = (
    F.when(etapa_funil == "Autosserviço", F.lit(1))
    .when(etapa_funil == "Atendimento Inicial", F.lit(2))
    .when(etapa_funil == "Atendimento Especializado", F.lit(3))
    .otherwise(F.lit(None).cast("int"))
)

df_dim_canal = (
    df_silver_canais
    .withColumn("SK_canal", F.row_number().over(w_sk).cast("bigint"))
    .withColumn("etapa_funil", etapa_funil)
    .withColumn("nivel_etapa", nivel_etapa)
    .select(
        F.col("SK_canal"),
        F.col("id_canal").cast("int").alias("id_canal"),
        F.col("nome_canal").cast("string").alias("nome"),
        F.col("tipo_canal").cast("string").alias("tipo"),
        F.col("etapa_funil").cast("string").alias("etapa_funil"),
        F.col("nivel_etapa").cast("int").alias("nivel_etapa"),
        F.col("status_canal").cast("string").alias("status"),
    )
    .orderBy("SK_canal")
)

display(df_dim_canal)

df_dim_canal.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_canal")

## gold.dim_atendente

In [0]:
#gold.dim_atendente (origem silver.tb_base_atendentes)

w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_atendente").cast("int"))

df_silver_atendentes = spark.read.table(f"{silver_schema}.tb_base_atendentes")

df_dim_atendente = (
    df_silver_atendentes
    .withColumn("SK_atendente", F.row_number().over(w_sk).cast("bigint"))
    .select(
        F.col("SK_atendente"),
        F.col("id_atendente").cast("int").alias("id_atendente"),
        F.col("nome_atendente").cast("string").alias("nome"),
        F.col("nivel_atendimento").cast("int").alias("nivel_atendimento"),
        F.col("data_admissao").cast("date").alias("data_admissao"),
        F.col("data_desligamento").cast("date").alias("data_desligamento"),
        F.col("status_atendente").cast("string").alias("status"),
    )
    .orderBy("SK_atendente")
)

display(df_dim_atendente)

df_dim_atendente.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_atendente")

##gold.dim_cliente

In [0]:
#gold.dim_cliente (origem silver.tb_clientes)

# id_cliente (CPF) fica string: com 11 dígitos, a ordem de texto coincide com a numérica
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_cliente"))

df_silver_clientes = spark.read.table(f"{silver_schema}.tb_clientes")

# idade já foi normalizada como inteiro na camada silver
idade = F.col("idade")

# maior quantidade de clientes se encontram na faixa entre 30-80 anos
# então buscamos maior granularidade por faixa de 10 anos entre 18 e 80 anos
faixa_idade = (
    F.when(idade.isNull() , F.lit("Não Informado"))
    .when(idade <= 24, F.lit("18-24"))
    .when(idade <= 34, F.lit("25-34"))
    .when(idade <= 44, F.lit("35-44"))
    .when(idade <= 54, F.lit("45-54"))
    .when(idade <= 44, F.lit("55-64"))
    .when(idade <= 54, F.lit("65-74"))
    .when(idade <= 64, F.lit("75-84"))
    .otherwise(F.lit("80+"))
)

df_dim_cliente = (
    df_silver_clientes
    .withColumn("SK_cliente", F.row_number().over(w_sk).cast("bigint"))
    .withColumn("faixa_idade", faixa_idade)
    .select(
        F.col("SK_cliente"),
        F.col("id_cliente").cast("string").alias("id_cliente"),
        F.col("nome").cast("string").alias("nome"),
        F.col("email").cast("string").alias("email"),
        F.col("regiao").cast("string").alias("regiao"),
        F.col("faixa_idade").cast("string").alias("faixa_idade"),
        F.col("data_cadastro").alias("data_cadastro"),
        F.expr("try_cast(dia_vencimento_fatura AS INT)").alias("dia_vencimento_fatura"),
    )
    .orderBy("SK_cliente")
)

display(df_dim_cliente)

df_dim_cliente.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_cliente")

##gold.dim_data


In [0]:
#gold.dim_data (calendário gerado; intervalo coberto pela silver.tb_chamados_hora)

df_silver_chamados_hora = spark.read.table(f"{silver_schema}.tb_chamados_hora")

# ---------- limites do calendário ----------
limites = df_silver_chamados_hora.agg(
    F.min(F.least("abertura_data", "inicio_data", "fim_data")).alias("min"),
    F.max(F.greatest("abertura_data", "inicio_data", "fim_data")).alias("max"),
).first()

if limites["min"] is None:
    raise ValueError("Nenhuma data válida encontrada em tb_chamados_hora")

data_inicio = date(limites["min"].year, 1, 1)
data_fim    = date(limites["max"].year, 12, 31)
print(f"Datas nos chamados: {limites['min']} a {limites['max']}")
print(f"Calendário gerado:  {data_inicio} a {data_fim}")

# ---------- calendário ----------
DIAS_SEMANA = ["Domingo", "Segunda-feira", "Terça-feira", "Quarta-feira",
               "Quinta-feira", "Sexta-feira", "Sábado"]          # dayofweek: 1 = domingo

df_calendario = spark.range(1).select(
    F.explode(F.sequence(F.lit(data_inicio), F.lit(data_fim), F.expr("INTERVAL 1 DAY"))).alias("data")
)

df_dim_data = df_calendario.select(
    (F.year("data") * 10000 + F.month("data") * 100 + F.dayofmonth("data")).cast("int").alias("SK_data"),
    F.col("data"),
    F.year("data").alias("ano"),
    F.month("data").alias("mes"),
    F.dayofmonth("data").alias("dia"),
    F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]), F.dayofweek("data")).alias("dia_semana"),
)

# ---------- membro "não informado" ----------
membro_desconhecido = spark.createDataFrame(
    [(-1, None, None, None, None, "Não Informado")],
    "SK_data int, data date, ano int, mes int, dia int, dia_semana string",
)
df_dim_data = df_dim_data.unionByName(membro_desconhecido).orderBy("SK_data")

display(df_dim_data)

df_dim_data.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_data")

##gold.dim_hora


In [0]:
#gold.dim_hora (gerada: todos os minutos do dia, de 00:00 a 23:59)

df_dim_hora = (
    spark.range(24 * 60)                                   # id = minuto do dia (0 a 1439)
    .withColumn("hora", (F.col("id") / 60).cast("int"))
    .withColumn("minuto", (F.col("id") % 60).cast("int"))
    .select(
        (F.col("hora") * 100 + F.col("minuto")).alias("SK_hora"),   # 09:23 -> 923
        "hora",
        "minuto",
    )
)

# ---------- membro "não informado" ----------
membro_desconhecido = spark.createDataFrame(
    [(-1, None, None)],
    "SK_hora int, hora int, minuto int",
)
df_dim_hora = df_dim_hora.unionByName(membro_desconhecido).orderBy("SK_hora")

display(df_dim_hora)

df_dim_hora.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_hora")

In [0]:
def mostrar(df_, descricao):
    if not df_.isEmpty():
        print(f"\n{descricao}")
        display(df_)

In [0]:
df_silver_chamados = spark.read.table(f"{silver_schema}.tb_chamados")

# lado da hora: id_cliente renomeado + marcador para saber se o chamado achou par no join
df_hora = (
    df_silver_chamados_hora
    .withColumnRenamed("id_cliente", "id_cliente_hora")
    .withColumn("_tem_hora", F.lit(True))
)

df_chamados_com_hora = df_silver_chamados.join(df_hora, "id_chamado", "left")

# ---------- 0. o join multiplicou linhas? ----------
n_chamados = df_silver_chamados.count()
n_join = df_chamados_com_hora.count()
print(f"Linhas em tb_chamados: {n_chamados} | após o join: {n_join}")
if n_join > n_chamados:
    print("  -> há id_chamado repetido em tb_chamados_hora: o join duplicou chamados")
else:
    print("  -> ok, não há duplicação")

# ---------- 1. id_cliente: chamados x chamados_hora ----------
c, h = F.col("id_cliente"), F.col("id_cliente_hora")
df_comp = df_chamados_com_hora.withColumn(
    "status_id_cliente",
    F.when(F.col("_tem_hora").isNull(), "sem registro em chamados_hora")
     .when(c.isNull() & h.isNull(), "nulo nos dois")
     .when(c.isNull(), "nulo só em chamados")
     .when(h.isNull(), "nulo só em chamados_hora")
     .when(c == h, "iguais")
     .otherwise("diferentes"),
)
print("\n=== id_cliente: chamados x chamados_hora ===")
display(df_comp.groupBy("status_id_cliente").count().orderBy(F.desc("count")))
mostrar(
    df_comp.filter(F.col("status_id_cliente") == "diferentes")
           .select("id_chamado", "id_cliente", "id_cliente_hora"),
    "Chamados com id_cliente diferente entre as duas tabelas",
)

# ---------- 2. chamados sem data de abertura ----------
sem_registro  = df_comp.filter(F.col("_tem_hora").isNull())
abertura_nula = df_comp.filter(F.col("_tem_hora") & F.col("abertura_data").isNull())

print("\n=== chamados sem data de abertura ===")
print(f"Sem nenhuma linha em chamados_hora: {sem_registro.count()}")
print(f"Com linha em chamados_hora, mas abertura_data nula: {abertura_nula.count()}")
mostrar(sem_registro.select("id_chamado", "id_cliente", "motivo", "canal"),
        "Chamados sem registro de horário")
mostrar(abertura_nula, "Chamados com registro de horário, mas abertura_data nula")

df_chamados_com_hora = df_chamados_com_hora.drop("_tem_hora", "id_cliente_hora")

##gold.dim_status_chamado

In [0]:
#gold.dim_status (todas as combinações possíveis de resolvido x abandono)

# ---------- regra de negócio: resolvido nulo = não resolvido ----------
df_chamados_com_hora = df_chamados_com_hora.withColumn(
    "resolvido", F.coalesce(F.col("resolvido"), F.lit(False))
)

# ---------- flag de abandono ----------
# abandono: o chamado foi aberto, não teve início de atendimento e não foi resolvido
df_chamados_com_hora = df_chamados_com_hora.withColumn(
    "abandono",
    (~F.col("resolvido"))
    & F.col("abertura_data").isNotNull()
    & F.col("inicio_data").isNull(),
)

print("Distribuição de resolvido x abandono nos chamados:")
display(df_chamados_com_hora.groupBy("resolvido", "abandono").count().orderBy("resolvido", "abandono"))

# ---------- dimensão ----------
df_dim_status = spark.createDataFrame(
    [
        (1, True,  False),     # resolvido
        (2, False, False),     # não resolvido, mas foi atendido
        (3, False, True),      # abandonado na espera
    ],
    "SK_status_chamado int, resolvido boolean, abandono boolean",
).orderBy("SK_status_chamado")

display(df_dim_status)

df_dim_status.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_status")